In [1]:
def main(datasources, start_date, end_date):
    """
    BigAlpha XGBoost factor with wider microstructure features,
    conservative tree complexity, compressed labels, and validation-only
    direction calibration.
    """
    import time
    import numpy as np
    import pandas as pd
    import dai
    import xgboost as xgb
    import structlog

    logger = structlog.get_logger()

    TRAIN_START = '2019-01-01 00:00:00'
    TRAIN_END = '2024-12-31 23:59:59'
    PROBE_TRAIN_END = '2023-12-31 23:59:59'
    PROBE_VALID_START = '2024-01-01 00:00:00'
    PROBE_VALID_END = '2024-12-31 23:59:59'
    PRICE_LOOKBACK_DAYS = 30
    FIN_LOOKBACK_DAYS = 430
    EPS = 1e-8

    micro_feature_cols = [
        'order_size_ratio_mean',
        'order_size_ratio_std',
        'deep_book_imbalance_mean',
        'deep_book_imbalance_std',
        'book_slope_mean',
        'book_slope_std',
        'relative_spread_mean',
        'relative_spread_std',
        'top_book_imbalance_mean',
        'top_book_imbalance_std',
        'deep_depth_log_mean',
        'deep_depth_log_std',
        'opening_order_size_ratio',
        'midday_order_size_ratio',
        'closing_order_size_ratio',
        'opening_deep_book_imbalance',
        'midday_deep_book_imbalance',
        'closing_deep_book_imbalance',
        'opening_book_slope',
        'midday_book_slope',
        'closing_book_slope',
        'opening_relative_spread',
        'midday_relative_spread',
        'closing_relative_spread',
        'opening_amount_ratio',
        'midday_amount_ratio',
        'closing_amount_ratio',
        'institutional_vol_ratio',
        'closing_big_order_flow',
        'closing_minus_open_order_size_ratio',
        'closing_minus_open_deep_book_imbalance',
        'closing_minus_midday_order_size_ratio',
        'midday_minus_open_order_size_ratio',
    ]

    rolling_base_cols = [
        'order_size_ratio_mean',
        'deep_book_imbalance_mean',
        'book_slope_mean',
        'relative_spread_mean',
        'closing_big_order_flow',
    ]
    rolling_suffixes = ['lag1', 'ma3', 'ma5', 'diff1', 'ma5_minus_ma3']
    rolling_feature_cols = [
        f'{col}_{suffix}'
        for col in rolling_base_cols
        for suffix in rolling_suffixes
    ]

    financial_feature_cols = [
        'asset_turnover',
        'roe_lf',
        'asset_turnover_roc63',
        'asset_turnover_roc252',
        'asset_turnover_z252',
        'roe_lf_roc63',
        'roe_lf_roc252',
        'roe_lf_z252',
        'asset_turnover_x_roe',
    ]

    feature_cols = list(dict.fromkeys(
        micro_feature_cols + rolling_feature_cols + financial_feature_cols
    ))

    def make_model():
        return xgb.XGBRegressor(
            n_estimators=80,
            max_depth=3,
            learning_rate=0.05,
            subsample=0.65,
            colsample_bytree=0.5,
            min_child_weight=30,
            reg_alpha=0.5,
            reg_lambda=8.0,
            max_bin=128,
            tree_method="hist",
            n_jobs=-1,
            random_state=42,
        )

    def safe_numeric(df, cols):
        for col in cols:
            if col not in df.columns:
                df[col] = np.nan
            df[col] = pd.to_numeric(df[col], errors='coerce')
            df[col] = df[col].replace([np.inf, -np.inf], np.nan)
        return df

    def fill_natural_dates(df, sd_str, ed_str):
        if df.empty:
            return df
        df = df.copy()
        df['date'] = pd.to_datetime(df['date'])
        natural_dates = pd.date_range(start=sd_str, end=ed_str, freq='D')
        filled_parts = []
        for instrument, group_df in df.sort_values('date').groupby('instrument'):
            group_df = group_df.drop_duplicates('date', keep='last').set_index('date')
            reindexed = group_df.reindex(natural_dates).ffill()
            reindexed['instrument'] = instrument
            filled_parts.append(reindexed.reset_index().rename(columns={'index': 'date'}))
        return pd.concat(filled_parts, ignore_index=True) if filled_parts else df.iloc[0:0]

    def build_financial_features(financial_table, sd, ed):
        fin_start = pd.to_datetime(sd) - pd.Timedelta(days=FIN_LOOKBACK_DAYS)
        fin_cols = [
            'operating_revenue',
            'net_profit_to_parent_shareholders',
            'total_assets',
            'total_equity_to_parent_shareholders',
        ]
        fin_select = ', '.join(fin_cols)
        fin_sql = f"""
        SELECT date, instrument, {fin_select}
        FROM {financial_table}
        WHERE category='lf' AND shift=0
        """
        fin = dai.query(fin_sql, filters={'date': [fin_start, ed]}).df()
        fin = fill_natural_dates(fin, fin_start, ed)
        fin['date'] = pd.to_datetime(fin['date'])
        fin['instrument'] = fin['instrument'].astype(str)
        fin = safe_numeric(fin, fin_cols)

        fin['asset_turnover'] = fin['operating_revenue'] / (fin['total_assets'] + EPS)
        fin['roe_lf'] = (
            fin['net_profit_to_parent_shareholders'] /
            (fin['total_equity_to_parent_shareholders'] + EPS)
        )

        fin = fin.sort_values(['instrument', 'date']).reset_index(drop=True)
        for base_col in ['asset_turnover', 'roe_lf']:
            group = fin.groupby('instrument', group_keys=False)[base_col]
            shifted_63 = group.shift(63)
            shifted_252 = group.shift(252)
            roll_mean_252 = group.transform(
                lambda s: s.rolling(252, min_periods=63).mean()
            )
            roll_std_252 = group.transform(
                lambda s: s.rolling(252, min_periods=63).std()
            )
            fin[f'{base_col}_roc63'] = fin[base_col] / (shifted_63.abs() + EPS) - 1.0
            fin[f'{base_col}_roc252'] = fin[base_col] / (shifted_252.abs() + EPS) - 1.0
            fin[f'{base_col}_z252'] = (fin[base_col] - roll_mean_252) / (roll_std_252 + EPS)

        fin['asset_turnover_x_roe'] = fin['asset_turnover'] * fin['roe_lf']
        fin = safe_numeric(fin, financial_feature_cols)
        return fin[['date', 'instrument'] + financial_feature_cols]

    def build_price_features(bar1m_table, sd, ed):
        price_start = pd.to_datetime(sd) - pd.Timedelta(days=PRICE_LOOKBACK_DAYS)
        price_sql = f"""
        WITH minute_features AS (
            SELECT
                date_trunc('day', date)::DATE AS trading_day,
                instrument,
                date,
                amount,
                close,
                LN(
                    ((bid_volume1 / (bid_num_orders1 + 1e-8)) + 1e-8) /
                    ((ask_volume1 / (ask_num_orders1 + 1e-8)) + 1e-8)
                ) AS order_size_ratio,
                (
                    bid_volume3 + bid_volume4 + bid_volume5 -
                    (ask_volume3 + ask_volume4 + ask_volume5)
                ) / (
                    bid_volume3 + bid_volume4 + bid_volume5 +
                    ask_volume3 + ask_volume4 + ask_volume5 + 1e-8
                ) AS deep_book_imbalance,
                (
                    (bid_volume1 - ask_volume1) /
                    (bid_volume1 + ask_volume1 + 1e-8)
                ) - (
                    (bid_volume5 - ask_volume5) /
                    (bid_volume5 + ask_volume5 + 1e-8)
                ) AS book_slope,
                (ask_price1 - bid_price1) /
                    (((ask_price1 + bid_price1) / 2.0) + 1e-8) AS relative_spread,
                (bid_volume1 - ask_volume1) /
                    (bid_volume1 + ask_volume1 + 1e-8) AS top_book_imbalance,
                LN(
                    bid_volume3 + bid_volume4 + bid_volume5 +
                    ask_volume3 + ask_volume4 + ask_volume5 + 1e-8
                ) AS deep_depth_log,
                CASE WHEN (
                    (EXTRACT(HOUR FROM date) = 9 AND EXTRACT(MINUTE FROM date) >= 30)
                    OR (EXTRACT(HOUR FROM date) = 10 AND EXTRACT(MINUTE FROM date) < 0)
                ) THEN 1 ELSE 0 END AS is_opening,
                CASE WHEN (
                    (EXTRACT(HOUR FROM date) = 10)
                    OR (EXTRACT(HOUR FROM date) = 11 AND EXTRACT(MINUTE FROM date) <= 30)
                    OR (EXTRACT(HOUR FROM date) = 13)
                    OR (EXTRACT(HOUR FROM date) = 14 AND EXTRACT(MINUTE FROM date) < 30)
                ) THEN 1 ELSE 0 END AS is_midday,
                CASE WHEN (
                    EXTRACT(HOUR FROM date) = 14 AND EXTRACT(MINUTE FROM date) >= 30
                ) THEN 1 ELSE 0 END AS is_closing
            FROM {bar1m_table}
            WHERE ask_price1 > 0
              AND bid_price1 > 0
              AND ask_volume1 >= 0
              AND bid_volume1 >= 0
              AND ask_num_orders1 >= 0
              AND bid_num_orders1 >= 0
        )
        SELECT
            trading_day,
            instrument,
            AVG(order_size_ratio) AS order_size_ratio_mean,
            SQRT(ABS(AVG(order_size_ratio * order_size_ratio) -
                AVG(order_size_ratio) * AVG(order_size_ratio))) AS order_size_ratio_std,
            AVG(deep_book_imbalance) AS deep_book_imbalance_mean,
            SQRT(ABS(AVG(deep_book_imbalance * deep_book_imbalance) -
                AVG(deep_book_imbalance) * AVG(deep_book_imbalance))) AS deep_book_imbalance_std,
            AVG(book_slope) AS book_slope_mean,
            SQRT(ABS(AVG(book_slope * book_slope) -
                AVG(book_slope) * AVG(book_slope))) AS book_slope_std,
            AVG(relative_spread) AS relative_spread_mean,
            SQRT(ABS(AVG(relative_spread * relative_spread) -
                AVG(relative_spread) * AVG(relative_spread))) AS relative_spread_std,
            AVG(top_book_imbalance) AS top_book_imbalance_mean,
            SQRT(ABS(AVG(top_book_imbalance * top_book_imbalance) -
                AVG(top_book_imbalance) * AVG(top_book_imbalance))) AS top_book_imbalance_std,
            AVG(deep_depth_log) AS deep_depth_log_mean,
            SQRT(ABS(AVG(deep_depth_log * deep_depth_log) -
                AVG(deep_depth_log) * AVG(deep_depth_log))) AS deep_depth_log_std,
            AVG(CASE WHEN is_opening = 1 THEN order_size_ratio END) AS opening_order_size_ratio,
            AVG(CASE WHEN is_midday = 1 THEN order_size_ratio END) AS midday_order_size_ratio,
            AVG(CASE WHEN is_closing = 1 THEN order_size_ratio END) AS closing_order_size_ratio,
            AVG(CASE WHEN is_opening = 1 THEN deep_book_imbalance END) AS opening_deep_book_imbalance,
            AVG(CASE WHEN is_midday = 1 THEN deep_book_imbalance END) AS midday_deep_book_imbalance,
            AVG(CASE WHEN is_closing = 1 THEN deep_book_imbalance END) AS closing_deep_book_imbalance,
            AVG(CASE WHEN is_opening = 1 THEN book_slope END) AS opening_book_slope,
            AVG(CASE WHEN is_midday = 1 THEN book_slope END) AS midday_book_slope,
            AVG(CASE WHEN is_closing = 1 THEN book_slope END) AS closing_book_slope,
            AVG(CASE WHEN is_opening = 1 THEN relative_spread END) AS opening_relative_spread,
            AVG(CASE WHEN is_midday = 1 THEN relative_spread END) AS midday_relative_spread,
            AVG(CASE WHEN is_closing = 1 THEN relative_spread END) AS closing_relative_spread,
            SUM(CASE WHEN is_opening = 1 THEN amount ELSE 0 END) /
                (SUM(amount) + 1e-8) AS opening_amount_ratio,
            SUM(CASE WHEN is_midday = 1 THEN amount ELSE 0 END) /
                (SUM(amount) + 1e-8) AS midday_amount_ratio,
            SUM(CASE WHEN is_closing = 1 THEN amount ELSE 0 END) /
                (SUM(amount) + 1e-8) AS closing_amount_ratio,
            SUM(CASE WHEN is_midday = 1 THEN amount ELSE 0 END) /
                (SUM(amount) + 1e-8) AS institutional_vol_ratio,
            SUM(CASE WHEN is_closing = 1 THEN amount * order_size_ratio ELSE 0 END) /
                (SUM(CASE WHEN is_closing = 1 THEN amount ELSE 0 END) + 1e-8) AS closing_big_order_flow,
            ARG_MAX(close, date) AS close
        FROM minute_features
        GROUP BY trading_day, instrument
        ORDER BY trading_day, instrument
        """
        price = dai.query(
            price_sql,
            filters={'date': [price_start, ed]},
            compression=True,
        ).df().rename(columns={'trading_day': 'date'})
        price['date'] = pd.to_datetime(price['date'])
        price['instrument'] = price['instrument'].astype(str)
        price = safe_numeric(price, micro_feature_cols + ['close'])

        price['closing_minus_open_order_size_ratio'] = (
            price['closing_order_size_ratio'] - price['opening_order_size_ratio']
        )
        price['closing_minus_open_deep_book_imbalance'] = (
            price['closing_deep_book_imbalance'] - price['opening_deep_book_imbalance']
        )
        price['closing_minus_midday_order_size_ratio'] = (
            price['closing_order_size_ratio'] - price['midday_order_size_ratio']
        )
        price['midday_minus_open_order_size_ratio'] = (
            price['midday_order_size_ratio'] - price['opening_order_size_ratio']
        )

        price = safe_numeric(price, micro_feature_cols + ['close'])
        price = price.sort_values(['instrument', 'date']).reset_index(drop=True)
        for col in rolling_base_cols:
            group = price.groupby('instrument', group_keys=False)[col]
            price[f'{col}_lag1'] = group.shift(1)
            price[f'{col}_ma3'] = group.transform(
                lambda s: s.rolling(3, min_periods=2).mean()
            )
            price[f'{col}_ma5'] = group.transform(
                lambda s: s.rolling(5, min_periods=3).mean()
            )
            price[f'{col}_diff1'] = price[col] - price[f'{col}_lag1']
            price[f'{col}_ma5_minus_ma3'] = price[f'{col}_ma5'] - price[f'{col}_ma3']

        price = safe_numeric(price, micro_feature_cols + rolling_feature_cols + ['close'])
        return price

    def add_labels(df):
        df = df.sort_values(['instrument', 'date']).reset_index(drop=True)
        df['raw_label'] = (
            df.groupby('instrument', group_keys=False)['close'].shift(-1) /
            df['close'] - 1.0
        )
        label_mean = df.groupby('date')['raw_label'].transform('mean')
        label_std = df.groupby('date')['raw_label'].transform('std')
        label_z = (df['raw_label'] - label_mean) / (label_std + EPS)
        label_z = label_z.clip(-5.0, 5.0)
        df['label'] = np.tanh(label_z / 3.0)
        df['label'] = df['label'].replace([np.inf, -np.inf], np.nan)
        return df

    def cross_section_zscore(df, cols):
        df = safe_numeric(df, cols)
        for col in cols:
            mean_series = df.groupby('date')[col].transform('mean')
            std_series = df.groupby('date')[col].transform('std')
            df[col] = (df[col] - mean_series) / (std_series + EPS)
            df[col] = df[col].replace([np.inf, -np.inf], np.nan).fillna(0.0)
        return df

    def build_features(financial_table, bar1m_table, sd, ed, with_label=True):
        t0 = time.time()
        logger.info("build_features start", start=str(sd), end=str(ed))

        price = build_price_features(bar1m_table, sd, ed)
        fin = build_financial_features(financial_table, sd, ed)
        df = pd.merge(price, fin, how='left', on=['date', 'instrument'])

        if with_label:
            df = add_labels(df)

        df = df[
            (df['date'] >= pd.to_datetime(sd)) &
            (df['date'] <= pd.to_datetime(ed))
        ].copy()

        stk_pool = dai.query(
            "SELECT date, instrument FROM bigalpha_2026_instruments",
            filters={'date': [sd, ed]},
        ).df()
        stk_pool['date'] = pd.to_datetime(stk_pool['date'])
        stk_pool['instrument'] = stk_pool['instrument'].astype(str)
        df['instrument'] = df['instrument'].astype(str)
        df = pd.merge(df, stk_pool, how='right', on=['date', 'instrument'])
        df = cross_section_zscore(df, feature_cols)
        logger.info("build_features done", rows=len(df), elapsed=round(time.time() - t0, 2))
        return df.reset_index(drop=True)

    def valid_training_rows(df):
        clean = df.dropna(subset=['label', 'instrument']).copy()
        clean = clean.replace([np.inf, -np.inf], np.nan)
        clean[feature_cols] = clean[feature_cols].fillna(0.0)
        return clean

    def daily_spearman_ic(df, pred_col, label_col):
        values = []
        for _, group in df[[pred_col, label_col, 'date']].dropna().groupby('date'):
            if len(group) < 20:
                continue
            if group[pred_col].nunique() < 2 or group[label_col].nunique() < 2:
                continue
            ic = group[pred_col].rank().corr(group[label_col].rank())
            if pd.notna(ic) and np.isfinite(ic):
                values.append(float(ic))
        return float(np.mean(values)) if values else 0.0

    logger.info("building full training set", train_start=TRAIN_START, train_end=TRAIN_END)
    train_full = build_features(
        'bigalpha_2026_financial',
        'bigalpha_2026_stock_bar1m',
        TRAIN_START,
        TRAIN_END,
        with_label=True,
    )
    train_full = valid_training_rows(train_full)

    probe_train = train_full[train_full['date'] <= pd.to_datetime(PROBE_TRAIN_END)].copy()
    probe_valid = train_full[
        (train_full['date'] >= pd.to_datetime(PROBE_VALID_START)) &
        (train_full['date'] <= pd.to_datetime(PROBE_VALID_END))
    ].copy()

    direction = 1.0
    if len(probe_train) > 0 and len(probe_valid) > 0:
        probe_model = make_model()
        probe_model.fit(probe_train[feature_cols], probe_train['label'])
        probe_valid['probe_factor'] = probe_model.predict(probe_valid[feature_cols])
        probe_ic_mean = daily_spearman_ic(probe_valid, 'probe_factor', 'raw_label')
        direction = -1.0 if probe_ic_mean < 0 else 1.0
        logger.info("probe direction calibrated", probe_ic_mean=round(probe_ic_mean, 6), direction=direction)
    else:
        logger.warning("probe direction skipped because validation split is empty")

    t_fit = time.time()
    model = make_model()
    model.fit(train_full[feature_cols], train_full['label'])
    logger.info("final model trained", samples=len(train_full), elapsed=round(time.time() - t_fit, 2))

    bar1m_table = datasources['bar1m']
    financial_table = datasources['financial']
    test_df = build_features(financial_table, bar1m_table, start_date, end_date, with_label=False)
    test_df[feature_cols] = test_df[feature_cols].fillna(0.0)
    test_df['factor'] = direction * model.predict(test_df[feature_cols])

    result = test_df[['date', 'instrument', 'factor']].copy()
    result['factor'] = pd.to_numeric(result['factor'], errors='coerce')
    result['factor'] = result['factor'].replace([np.inf, -np.inf], np.nan)
    result['factor'] = result.groupby('date')['factor'].transform(
        lambda s: s.fillna(s.median())
    )
    result['factor'] = result['factor'].fillna(0.0)
    result = result.dropna(subset=['date', 'instrument']).reset_index(drop=True)
    result = result[['date', 'instrument', 'factor']]
    logger.info("factor production finished", rows=len(result), features=len(feature_cols))
    return result


if __name__ == '__main__':
    from bigmodule import M
    import dai
    import structlog

    logger = structlog.get_logger()
    datasources = {
        'bar1m': 'bigalpha_2026_stock_bar1m',
        'financial': 'bigalpha_2026_financial',
    }
    start_date = '2024-01-01 00:00:00'
    end_date = '2024-12-31 23:59:59'

    logger.info(f"计算因子，测试区间：{start_date} ~ {end_date}")
    factor_data = main(datasources, start_date, end_date)

    logger.info(f"读取因子库，区间：{start_date} ~ {end_date}")
    factor_pool = dai.query(
        "SELECT * FROM bigalpha_2026_factorlib",
        filters={'date': [start_date, end_date]},
    ).df()

    result = M.bigalpha_eval._latest(
        factor_data=factor_data,
        factor_pool=factor_pool,
        process_pools=False,
        show=True,
    )


[2026-07-06 15:24:51] [info     ] 计算因子，测试区间：2024-01-01 00:00:00 ~ 2024-12-31 23:59:59
